# RecurrentBitNet V2 — Production Training

Full-scale pre-training on FineWeb-Edu with:
- **Progressive recurrence curriculum** (R=1→2→3→4 over 500K steps)
- **Auxiliary per-recurrence loss** with exponential decay
- **Differential learning rates** per module group
- **Google Drive checkpointing** for Colab disconnect protection
- **Checkpoint resume** — restart without losing progress

**Architecture** (from `DESIGN.md`):
```
Encoder (3 blocks) → Reasoning Core (6 blocks × R) → Decoder (3 blocks)
```

**Expected runtime**: ~14-20 hours on H100 for 500K steps.

## 1. Setup: Install Dependencies & Mount Google Drive

In [ ]:
# Install dependencies
!pip install -q datasets transformers tqdm matplotlib

# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

# Checkpoint directories
import os
DRIVE_CKPT_DIR = '/content/drive/MyDrive/recurrent_bitnet_v2'
LOCAL_CKPT_DIR = '/content/checkpoints_v2'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
print(f"Drive checkpoints → {DRIVE_CKPT_DIR}")
print(f"Local checkpoints → {LOCAL_CKPT_DIR}")

## 2. Environment & Device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import shutil
from dataclasses import dataclass, asdict
from typing import Optional
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_mem / 1e9
    print(f"GPU: {props.name} — {VRAM_GB:.1f} GB VRAM")
    print(f"Compute Capability: {props.major}.{props.minor}")
    if hasattr(torch.backends, 'cuda'):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("TF32 enabled for matmul + cudnn")

## 3. BitLinear (1.58-bit Ternary Quantization)

Ternary weights {-1, 0, +1} with STE. Full-precision latent weights for gradient updates.

In [ ]:
def ste_round(x: torch.Tensor) -> torch.Tensor:
    return x + (x.round() - x).detach()

def quantize_weights_ternary(w: torch.Tensor):
    scale = w.abs().mean().clamp(min=1e-5)
    w_normalized = w / scale
    w_ternary = ste_round(w_normalized).clamp(-1, 1)
    return w_ternary, scale

def quantize_activations_int8(x: torch.Tensor):
    Qb = 127
    scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_int = (x * Qb / scale).round().clamp(-Qb, Qb)
    return x_int, scale


class BitLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__(in_features, out_features, bias=bias)

    @torch.amp.custom_fwd(device_type="cuda")
    def forward(self, x):
        w_ternary, w_scale = quantize_weights_ternary(self.weight)
        w_effective = self.weight + (w_ternary * w_scale - self.weight).detach()
        x_int, x_scale = quantize_activations_int8(x)
        x_effective = x + (x_int * x_scale / 127.0 - x).detach()
        return x_effective @ w_effective.t()

## 4. Architecture Components

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (x_fp32 * norm).to(x.dtype) * self.weight


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = BitLinear(d_model, 3 * d_model, bias=False)
        self.out_proj = BitLinear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, L, D = x.size()
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        # Use PyTorch scaled_dot_product_attention for Flash Attention on H100
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=(mask is None)
        )
        return self.out_proj(out.transpose(1, 2).reshape(B, L, D))


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = BitLinear(d_model, d_ff, bias=False)
        self.w2 = BitLinear(d_ff, d_model, bias=False)
        self.w3 = BitLinear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## 5. Encoder / Reasoning Core / Decoder

- **Encoder**: Parses input format → abstract representation (non-recurrent)
- **Reasoning Core**: Recurrent blocks × R with iteration embeddings + dropout
- **Decoder**: Abstract representation → output tokens (non-recurrent)

In [ ]:
class EncoderStack(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff)
            for _ in range(config.encoder_blocks)
        ])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x


class ReasoningCore(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff)
            for _ in range(config.reasoning_blocks)
        ])
        max_R = config.max_recurrence
        self.iteration_embeddings = nn.Parameter(
            torch.randn(max_R, 1, 1, config.d_model) * 0.02
        )
        self.halt_scorer = nn.Sequential(
            nn.Linear(config.d_model, 1), nn.Sigmoid()
        )

    def forward(self, x, mask=None, R=None, recurrence_dropout=0.0):
        if R is None:
            R = self.iteration_embeddings.size(0)
        iter_outputs = []
        for r in range(R):
            if self.training and recurrence_dropout > 0 and r > 0:
                if torch.rand(1).item() < recurrence_dropout:
                    continue
            if r < self.iteration_embeddings.size(0):
                x = x + self.iteration_embeddings[r]
            for block in self.blocks:
                x = block(x, mask)
            iter_outputs.append(x)
        return x, iter_outputs


class DecoderStack(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff)
            for _ in range(config.decoder_blocks)
        ])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x

## 6. Model Assembly & Configuration

In [ ]:
@dataclass
class ModelConfig:
    # --- Architecture ---
    d_model: int = 768
    n_heads: int = 12
    d_ff: int = 3072
    vocab_size: int = 32000
    max_seq_len: int = 1024

    # --- Structure (from DESIGN.md) ---
    encoder_blocks: int = 3
    reasoning_blocks: int = 6
    max_recurrence: int = 4      # Maximum R (curriculum target)
    decoder_blocks: int = 3
    recurrence_dropout: float = 0.1


class RecurrentBitNetV2(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = EncoderStack(config)
        self.reasoning_core = ReasoningCore(config)
        self.decoder = DecoderStack(config)
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = BitLinear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying

    def forward(self, idx, targets=None, R=None):
        B, L = idx.size()
        x = self.token_emb(idx)

        x = self.encoder(x)
        x, iter_outputs = self.reasoning_core(
            x, R=R,
            recurrence_dropout=self.config.recurrence_dropout if self.training else 0.0
        )
        x = self.decoder(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, iter_outputs


# --- Instantiate ---
config = ModelConfig()
model = RecurrentBitNetV2(config).to(DEVICE)

# Enable gradient checkpointing
for module in model.modules():
    if hasattr(module, 'gradient_checkpointing_enable'):
        module.gradient_checkpointing_enable()

num_params = sum(p.numel() for p in model.parameters())
eff_depth = config.encoder_blocks + config.reasoning_blocks * config.max_recurrence + config.decoder_blocks
print(f"✅ RecurrentBitNet V2")
print(f"   Unique parameters: {num_params:,}")
print(f"   Effective depth:   {eff_depth} layers (R={config.max_recurrence})")
print(f"   Est. inference:    ~{num_params * 0.25 / 1e6:.0f} MB at TQ2_0")

## 7. Training Configuration

All hyperparameters for the 500K-step production run.
Curriculum follows DESIGN.md §1: progressive recurrence depth.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TRAINING HYPERPARAMETERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TOTAL_STEPS   = 500_000
BATCH_SIZE    = 8
SEQ_LEN       = config.max_seq_len  # 1024
MAX_GRAD_NORM = 1.0
WARMUP_STEPS  = 2_000
PEAK_LR       = 2e-3
MIN_LR_RATIO  = 0.1                 # Cosine decays to 10% of peak
AUX_DECAY     = 0.3                 # α in auxiliary loss

# Logging & saving
LOG_EVERY     = 100
EVAL_EVERY    = 25_000
SAVE_LOCAL     = 5_000               # Save to /content every 5K steps
SAVE_DRIVE     = 25_000              # Copy to Drive every 25K steps

# Progressive recurrence curriculum (DESIGN.md §1)
CURRICULUM = [
    (0,       1),    # Steps 0-50K:    R=1 (learn basic representations)
    (50_000,  2),    # Steps 50K-150K: R=2 (learn to refine)
    (150_000, 3),    # Steps 150K-300K: R=3 (deeper refinement)
    (300_000, 4),    # Steps 300K+:    R=4 (full depth)
]

# Resume from checkpoint (set to path to resume, or None for fresh start)
RESUME_FROM = None
# Example: RESUME_FROM = '/content/drive/MyDrive/recurrent_bitnet_v2/checkpoint_step100000.pt'

total_tokens = TOTAL_STEPS * BATCH_SIZE * SEQ_LEN
print(f"Training plan:")
print(f"  Steps:          {TOTAL_STEPS:,}")
print(f"  Batch × SeqLen: {BATCH_SIZE} × {SEQ_LEN} = {BATCH_SIZE * SEQ_LEN:,} tokens/step")
print(f"  Total tokens:   {total_tokens / 1e9:.1f}B")
print(f"  Curriculum:     {CURRICULUM}")
print(f"  Checkpoints:    local every {SAVE_LOCAL:,}, Drive every {SAVE_DRIVE:,}")

## 8. Data Pipeline — FineWeb-Edu Streaming

Streams `HuggingFaceFW/fineweb-edu` (10BT sample) with automatic restart on exhaustion.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

PAD_ID = tokenizer.pad_token_id

fineweb = load_dataset(
    'HuggingFaceFW/fineweb-edu', name='sample-10BT',
    split='train', streaming=True
)
stream_iter = iter(fineweb)


def get_batch():
    global stream_iter
    inputs, targets = [], []
    while len(inputs) < BATCH_SIZE:
        try:
            text = next(stream_iter)['text']
        except StopIteration:
            stream_iter = iter(fineweb)
            text = next(stream_iter)['text']

        tokens = tokenizer(
            text, truncation=True, max_length=SEQ_LEN + 1, return_tensors='pt'
        )['input_ids'][0]

        # Skip very short documents (< 64 tokens) for quality
        if len(tokens) < 64:
            continue

        if len(tokens) < SEQ_LEN + 1:
            pad = torch.full((SEQ_LEN + 1 - len(tokens),), PAD_ID, dtype=torch.long)
            tokens = torch.cat([tokens, pad])

        inputs.append(tokens[:SEQ_LEN])
        targets.append(tokens[1:SEQ_LEN + 1])

    return torch.stack(inputs).to(DEVICE), torch.stack(targets).to(DEVICE)


# Verify
test_in, test_tgt = get_batch()
print(f"✅ Data pipeline ready")
print(f"   Batch shape: {test_in.shape}")
print(f"   Tokenizer vocab: {len(tokenizer):,}")
del test_in, test_tgt

## 9. Optimizer — Differential Learning Rates

Per DESIGN.md §2:
- Reasoning Core: **2× LR** (most reused weights, needs faster adaptation)
- Encoder/Decoder: standard LR
- Embeddings: **0.5× LR** (large, slow-moving)

In [ ]:
param_groups = [
    {'params': list(model.encoder.parameters()) + list(model.decoder.parameters()),
     'lr': PEAK_LR, 'name': 'encoder_decoder'},
    {'params': list(model.reasoning_core.parameters()),
     'lr': PEAK_LR * 2, 'name': 'reasoning_core'},
    {'params': [model.token_emb.weight],
     'lr': PEAK_LR * 0.5, 'name': 'embeddings'},
    {'params': [model.final_norm.weight],
     'lr': PEAK_LR, 'name': 'final_norm'},
]

optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95), weight_decay=0.1)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

# Resume from checkpoint
start_step = 0
loss_history = []
recurrence_history = []
eval_log = []
best_loss = float('inf')

if RESUME_FROM and os.path.exists(RESUME_FROM):
    print(f"🔄 Resuming from {RESUME_FROM}...")
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_step = ckpt['step']
    loss_history = ckpt.get('loss_history', [])
    recurrence_history = ckpt.get('recurrence_history', [])
    eval_log = ckpt.get('eval_log', [])
    best_loss = ckpt.get('best_loss', float('inf'))
    print(f"   Resumed at step {start_step:,}, best_loss={best_loss:.4f}")
else:
    print("🆕 Starting fresh training run")

print(f"\nOptimizer groups:")
for g in param_groups:
    n = sum(p.numel() for p in g['params'])
    print(f"  {g['name']}: {n:,} params @ lr={g['lr']:.1e}")

## 10. Evaluation Function

In [ ]:
@torch.no_grad()
def evaluate(model, num_batches=50, R=None):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    if R is None:
        R = config.max_recurrence

    for _ in tqdm(range(num_batches), desc=f"Eval (R={R})", leave=False):
        idx, targets = get_batch()
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda'), dtype=torch.bfloat16):
            logits, _, _ = model(idx, targets, R=R)

        logits_flat = logits.view(-1, logits.size(-1))
        targets_flat = targets.view(-1)
        mask = targets_flat != PAD_ID
        loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=PAD_ID, reduction='sum')
        total_loss += loss.item()
        total_tokens += mask.sum().item()

    model.train()
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(avg_loss, 100))
    return {'loss': avg_loss, 'perplexity': ppl, 'R': R}

print("✅ Evaluation function ready")

## 11. Checkpoint Save/Copy Functions

In [ ]:
def save_checkpoint(step, to_drive=False):
    """Save checkpoint locally and optionally copy to Drive."""
    ckpt = {
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'config': asdict(config),
        'loss_history': loss_history,
        'recurrence_history': recurrence_history,
        'eval_log': eval_log,
        'best_loss': best_loss,
    }
    # Local save
    local_path = os.path.join(LOCAL_CKPT_DIR, f'checkpoint_step{step}.pt')
    torch.save(ckpt, local_path)
    print(f"  💾 Local: {local_path}")

    if to_drive:
        drive_path = os.path.join(DRIVE_CKPT_DIR, f'checkpoint_step{step}.pt')
        shutil.copy2(local_path, drive_path)
        print(f"  ☁️  Drive: {drive_path}")

    # Keep only latest 3 local checkpoints to save disk
    local_ckpts = sorted([
        f for f in os.listdir(LOCAL_CKPT_DIR) if f.startswith('checkpoint_step')
    ])
    while len(local_ckpts) > 3:
        os.remove(os.path.join(LOCAL_CKPT_DIR, local_ckpts.pop(0)))

print("✅ Checkpoint functions ready")

## 12. Training Loop

Production loop with:
- BF16 mixed precision
- Progressive recurrence curriculum
- Auxiliary per-recurrence loss (L_total = L_final + Σ α^(R-r) × L_step_r)
- Periodic eval & Drive checkpointing
- ETA estimation

In [ ]:
model.train()
print("🚀 Starting production training...")
print(f"   Steps {start_step+1:,} → {TOTAL_STEPS:,}")
print("=" * 70)

run_start = time.time()
window_start = time.time()
window_loss = 0.0

for step in range(start_step + 1, TOTAL_STEPS + 1):
    t0 = time.time()

    # 1. Curriculum
    R = 1
    for threshold, depth in reversed(CURRICULUM):
        if step >= threshold:
            R = depth
            break

    # 2. Get batch
    idx, targets = get_batch()

    # 3. Forward
    with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda'), dtype=torch.bfloat16):
        logits, base_loss, iter_outputs = model(idx, targets, R=R)

        # 4. Auxiliary loss
        aux_loss = torch.tensor(0.0, device=DEVICE)
        for r, hidden in enumerate(iter_outputs):
            step_normed = model.final_norm(hidden)
            step_logits = model.lm_head(step_normed)
            step_loss = F.cross_entropy(
                step_logits.view(-1, step_logits.size(-1)), targets.view(-1)
            )
            aux_loss = aux_loss + (AUX_DECAY ** (R - (r + 1))) * step_loss

        total_loss = base_loss + aux_loss

    # 5. Backward
    optimizer.zero_grad()
    scaler.scale(total_loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    # 6. Track
    loss_val = total_loss.item()
    loss_history.append(loss_val)
    recurrence_history.append(R)
    window_loss += loss_val

    if loss_val < best_loss:
        best_loss = loss_val

    # 7. Log
    if step % LOG_EVERY == 0:
        elapsed = time.time() - window_start
        avg_loss = window_loss / LOG_EVERY
        ms_per_step = elapsed / LOG_EVERY * 1000
        lr = scheduler.get_last_lr()[0]
        remaining = (TOTAL_STEPS - step) * (elapsed / LOG_EVERY)
        eta_h = remaining / 3600

        print(
            f"Step {step:>7,}/{TOTAL_STEPS:,} | "
            f"Loss {avg_loss:.4f} | "
            f"R={R} | "
            f"LR {lr:.2e} | "
            f"{ms_per_step:.0f} ms/step | "
            f"ETA {eta_h:.1f}h"
        )
        window_start = time.time()
        window_loss = 0.0

    # 8. Save (local)
    if step % SAVE_LOCAL == 0:
        save_checkpoint(step, to_drive=False)

    # 9. Save (Drive) + Evaluate
    if step % SAVE_DRIVE == 0:
        save_checkpoint(step, to_drive=True)

    if step % EVAL_EVERY == 0:
        result = evaluate(model, num_batches=100, R=R)
        eval_log.append({'step': step, **result})
        print(
            f"  📊 Eval @ step {step:,}: "
            f"Loss={result['loss']:.4f}, PPL={result['perplexity']:.2f} (R={R})"
        )
        model.train()

total_time = time.time() - run_start
print("=" * 70)
print(f"✅ Training complete!")
print(f"   Total time: {total_time/3600:.1f} hours")
print(f"   Best loss:  {best_loss:.4f}")
print(f"   Final eval: {eval_log[-1] if eval_log else 'N/A'}")

## 13. Save Final Model to Drive

In [ ]:
# Save final model
save_checkpoint(TOTAL_STEPS, to_drive=True)

# Export ternary weights
print("\n📦 Exporting ternary weights...")
ternary_weights = {}
with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, BitLinear) and module.weight is not model.token_emb.weight:
            w_ternary, w_scale = quantize_weights_ternary(module.weight)
            ternary_weights[name] = {
                'weight_ternary': w_ternary.to(torch.int8).cpu(),
                'weight_scale': w_scale.float().cpu(),
            }

export_path = os.path.join(DRIVE_CKPT_DIR, 'ternary_export.pt')
torch.save(ternary_weights, export_path)
ternary_count = sum(v['weight_ternary'].numel() for v in ternary_weights.values())
print(f"📦 Ternary export → {export_path}")
print(f"   {ternary_count:,} ternary params → ~{ternary_count * 0.25 / 1e6:.1f} MB at TQ2_0")

# Save config as JSON for reference
import json
config_path = os.path.join(DRIVE_CKPT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(asdict(config), f, indent=2)
print(f"📋 Config → {config_path}")

## 14. Final Evaluation

Evaluates at each recurrence depth to measure the accuracy-speed tradeoff.

In [ ]:
print("📊 Final Evaluation — Recurrence Depth Comparison")
print("=" * 50)
for test_R in range(1, config.max_recurrence + 1):
    result = evaluate(model, num_batches=100, R=test_R)
    print(f"  R={test_R}: Loss={result['loss']:.4f}, Perplexity={result['perplexity']:.2f}")

print("\n(Lower R = faster inference, higher R = better quality)")
print("This demonstrates the accuracy-speed tradeoff from DESIGN.md.")

## 15. Training Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Loss curve
axes[0].plot(loss_history, alpha=0.1, color='steelblue')
if len(loss_history) > 100:
    window = min(500, len(loss_history) // 10)
    smoothed = []
    running = sum(loss_history[:window])
    for i in range(window, len(loss_history)):
        smoothed.append(running / window)
        running += loss_history[i] - loss_history[i - window]
    smoothed.append(running / window)
    axes[0].plot(range(window, len(loss_history) + 1), smoothed,
                 color='steelblue', linewidth=2)
axes[0].set_ylabel("Loss")
axes[0].set_title("RecurrentBitNet V2 — Training Loss")
axes[0].grid(alpha=0.3)

# Curriculum
axes[1].step(range(len(recurrence_history)), recurrence_history,
             where='post', color='coral', linewidth=2)
axes[1].set_ylabel("R")
axes[1].set_title("Progressive Recurrence Curriculum")
axes[1].set_yticks([1, 2, 3, 4])
axes[1].grid(alpha=0.3)

# Eval perplexity over time
if eval_log:
    eval_steps = [e['step'] for e in eval_log]
    eval_ppls = [e['perplexity'] for e in eval_log]
    axes[2].plot(eval_steps, eval_ppls, 'o-', color='green', linewidth=2, markersize=6)
    axes[2].set_ylabel("Perplexity")
    axes[2].set_title("Evaluation Perplexity Over Training")
    axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'No eval data yet', ha='center', va='center',
                 transform=axes[2].transAxes)

axes[-1].set_xlabel("Training Step")
plt.tight_layout()

# Save to Drive
plot_path = os.path.join(DRIVE_CKPT_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"📈 Saved → {plot_path}")